In [ ]:
import json
import librosa
import librosa.display
import random
import os

import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from collections import defaultdict
from IPython.display import clear_output
from PIL import Image
from scipy.io import wavfile
from scipy.spatial.distance import euclidean

In [ ]:
# grabbing stationarity scores for sounds that are in the in the eval set of AudioSet (which is what we typically use for memory experiment)
stationarity_scores_path = "/net/vast-storage/scratch/vast/mcdermott/yuecli/documents/things_playground/subimages_data.json"
#stationarity_scores_ = pd.read_csv(stationarity_scores_path)

with open(stationarity_scores_path) as f:
    stationarity_scores_ = json.load(f)

In [ ]:
stationarity_scores_[0]

In [ ]:
x = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_visual_v1/filenames.json"

# Open and read the JSON file
with open(x, 'r') as file:
    data = json.load(file)

# Print the data
print(data[0])

In [ ]:
# data = np.load("/net/vast-storage/scratch/vast/mcdermott/yuecli/documents/things_playground/subimgs_scores.npy")
# print(data)

In [ ]:
base_path = "/net/vast-storage/scratch/vast/mcdermott/yuecli/documents/things_playground/subimages/"

In [ ]:
im_look = 3

Could also just pick the most stationary image (and then pick the next most stationary image)
Could pick image that is opposing to the most stationary image 
(if the most stationary image is in the bottom left, then grab the top right image. If it is the top left, pick bottom right, etc)


In [ ]:
# two images at random

for mem_im_idx, im in enumerate(stationarity_scores_):
    print(mem_im_idx, im)
    # sample WITHOUT replacement
    rand_samp = random.sample(set([0, 1, 2, 3]), 2)

    print(rand_samp[0], rand_samp[1])

    image1 = np.asarray(Image.open(base_path + im['subimages'][rand_samp[0]]))
    image2 = np.asarray(Image.open(base_path + im['subimages'][rand_samp[1]]))

    to_find = "/".join(im['path'].split("/")[-2:])

    for entry in data:
        if to_find in entry['stim_path']:
            full_im_path = entry['orig_path']

    image  = np.asarray(Image.open(full_im_path))

    #Create a figure and axes to plot three images side by side
    fig, ax = plt.subplots(3, 3, figsize=(10, 10))  # 1 row, 2 columns
    
    # Plot the first image
    ax[0,0].imshow(image1, cmap='gray')
    ax[0,0].set_title(f'Random Image {rand_samp[0]}')
    
    # Plot the second image
    ax[0,1].imshow(image2, cmap='gray')
    ax[0, 1].set_title(f'Random Image {rand_samp[1]}')

    # Plot the full image
    ax[0, 2].imshow(image, cmap='gray')
    ax[0, 2].set_title('Full Image')

    # Define the directory where you want to save the images
    save_directory = "/om2/user/bjmedina/data/texture_pairs_visual/random"  # Change this to your desired path
    os.makedirs(save_directory, exist_ok=True)  # Ensure the directory exists
    
    # Save image1
    image1_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_0.png")
    Image.fromarray(image1).save(image1_path)
    
    # Save image2
    image2_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_1.png")
    Image.fromarray(image2).save(image2_path)
    
    print(f"Images saved to:\n{image1_path}\n{image2_path}")


    # sample WITHOUT replacement
    most_stationary_idx = np.argmin(im['scores'])
    diagonal_idx = {0: 3, 1: 2, 2: 1, 3: 0}[most_stationary_idx]

    print(most_stationary_idx, diagonal_idx)

    image1 = np.asarray(Image.open(base_path + im['subimages'][most_stationary_idx]))
    image2 = np.asarray(Image.open(base_path + im['subimages'][diagonal_idx]))

    to_find = "/".join(im['path'].split("/")[-2:])

    for entry in data:
        if to_find in entry['stim_path']:
            full_im_path = entry['orig_path']

    image  = np.asarray(Image.open(full_im_path))

    #Create a figure and axes to plot three images side by side
    
    # Plot the first image
    ax[1, 0].imshow(image1, cmap='gray')
    ax[1, 0].set_title(f'Image {most_stationary_idx} (most stationary)')
    
    # Plot the second image
    ax[1, 1].imshow(image2, cmap='gray')
    ax[1, 1].set_title(f'Image {diagonal_idx} (diagonal to most stationary)')

    # Plot the full image
    ax[1, 2].imshow(image, cmap='gray')
    ax[1, 2].set_title('Full Image')


    # Define the directory where you want to save the images
    save_directory = "/om2/user/bjmedina/data/texture_pairs_visual/stationary_diagonal"   # Change this to your desired path
    os.makedirs(save_directory, exist_ok=True)  # Ensure the directory exists
    
    # Save image1
    image1_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_0.png")
    Image.fromarray(image1).save(image1_path)
    
    # Save image2
    image2_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_1.png")
    Image.fromarray(image2).save(image2_path)
    
    print(f"Images saved to:\n{image1_path}\n{image2_path}")


    #Create a figure and axes to plot three images side by side
    # sample WITHOUT replacement
    sorted_indices = np.argsort(im['scores'])  # Sort indices by stationarity score (ascending order)
    
    most_stationary_idx = sorted_indices[0]  # Lowest (most stationary)
    second_most_stationary_idx = sorted_indices[1]  # Second lowest

    print(most_stationary_idx, second_most_stationary_idx)

    image1 = np.asarray(Image.open(base_path + im['subimages'][most_stationary_idx]))
    image2 = np.asarray(Image.open(base_path + im['subimages'][second_most_stationary_idx]))
    
    # Plot the first image
    ax[2,0].imshow(image1, cmap='gray')
    ax[2,0].set_title(f'Image {most_stationary_idx} (most stationary)')
    
    # Plot the second image
    ax[2,1].imshow(image2, cmap='gray')
    ax[2,1].set_title(f'Image {second_most_stationary_idx} (second most stationary)')

    # Plot the full image
    ax[2,2].imshow(image, cmap='gray')
    ax[2,2].set_title('Full Image')

    # Define the directory where you want to save the images
    save_directory = "/om2/user/bjmedina/data/texture_pairs_visual/most_stationary"   # Change this to your desired path
    os.makedirs(save_directory, exist_ok=True)  # Ensure the directory exists
    
    # Save image1
    image1_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_0.png")
    Image.fromarray(image1).save(image1_path)
    
    # Save image2
    image2_path = os.path.join(save_directory, f"mem_stim_{mem_im_idx}_1.png")
    Image.fromarray(image2).save(image2_path)
    
    print(f"Images saved to:\n{image1_path}\n{image2_path}")
        
    # Display the images
    plt.tight_layout()
    #plt.show()

    fig.savefig(f'/om2/user/bjmedina/data/texture_pairs_visual/summary/mem_stim_{mem_im_idx}.png')

    clear_output(wait=False)

In [ ]:
# two images (most stationary image, and its opposing/diagonal)

for im in stationarity_scores_[im_look:]:
    print(im['scores'])
    print(np.argmin(im['scores']))

    # sample WITHOUT replacement
    most_stationary_idx = np.argmin(im['scores'])
    diagonal_idx = {0: 3, 1: 2, 2: 1, 3: 0}[most_stationary_idx]

    print(most_stationary_idx, diagonal_idx)

    image1 = np.asarray(Image.open(base_path + im['subimages'][most_stationary_idx]))
    image2 = np.asarray(Image.open(base_path + im['subimages'][diagonal_idx]))

    to_find = "/".join(im['path'].split("/")[-2:])

    for entry in data:
        if to_find in entry['stim_path']:
            full_im_path = entry['orig_path']

    image  = np.asarray(Image.open(full_im_path))

    #Create a figure and axes to plot three images side by side
    fig, ax = plt.subplots(1, 3, figsize=(20, 10))  # 1 row, 2 columns
    
    # Plot the first image
    ax[0].imshow(image1, cmap='gray')
    ax[0].set_title(f'Image {most_stationary_idx} (most stationary)')
    
    # Plot the second image
    ax[1].imshow(image2, cmap='gray')
    ax[1].set_title(f'Image {diagonal_idx} (diagonal)')

    # Plot the full image
    ax[2].imshow(image, cmap='gray')
    ax[2].set_title('Full Image')
    
    # Display the images
    plt.tight_layout()
    plt.show()

    break

In [ ]:
# two images (most stationary image, and its opposing/diagonal)

for im in stationarity_scores_[im_look:]:
    print(im['scores'])
    print(np.argmin(im['scores']))

    # sample WITHOUT replacement
    sorted_indices = np.argsort(im['scores'])  # Sort indices by stationarity score (ascending order)
    
    most_stationary_idx = sorted_indices[0]  # Lowest (most stationary)
    second_most_stationary_idx = sorted_indices[1]  # Second lowest

    print(most_stationary_idx, second_most_stationary_idx)

    image1 = np.asarray(Image.open(base_path + im['subimages'][most_stationary_idx]))
    image2 = np.asarray(Image.open(base_path + im['subimages'][second_most_stationary_idx]))

    to_find = "/".join(im['path'].split("/")[-2:])

    for entry in data:
        if to_find in entry['stim_path']:
            full_im_path = entry['orig_path']

    image  = np.asarray(Image.open(full_im_path))

    #Create a figure and axes to plot three images side by side
    fig, ax = plt.subplots(1, 3, figsize=(20, 10))  # 1 row, 2 columns
    
    # Plot the first image
    ax[0].imshow(image1, cmap='gray')
    ax[0].set_title(f'Image {most_stationary_idx} (most stationary)')
    
    # Plot the second image
    ax[1].imshow(image2, cmap='gray')
    ax[1].set_title(f'Image {second_most_stationary_idx} (second most)')

    # Plot the full image
    ax[2].imshow(image, cmap='gray')
    ax[2].set_title('Full Image')
    
    # Display the images
    plt.tight_layout()
    plt.show()

    break

In [ ]:
ss_scores_to_texture = defaultdict(list)
times_to_texture     = defaultdict(list)

for texture_pair in texture_pairs:
    texture_id = "/".join(texture_pair['full_path'].split("/")[-2:])

    avg_ss_scores = stationarity_scores_[stationarity_scores_.filepath==texture_id].stationarity_score.tolist()
    times = stationarity_scores_[stationarity_scores_.filepath==texture_id].onset_time.tolist()
    
    ss_scores_to_texture[texture_id] = avg_ss_scores
    times_to_texture[texture_id]     = times

In [ ]:
# plot distribution of stationarity scores
avg_scores = []
for id in ss_scores_to_texture:
    avg_scores.extend(ss_scores_to_texture[id])

In [ ]:
plt.title("Stationarity Scores")
plt.axvline(x=np.mean(avg_scores), label=f"$\mu={np.mean(avg_scores)}$")
plt.hist(avg_scores, bins=100, alpha=0.5)
plt.legend()

avg_ss_score = np.mean(avg_scores)
std_ss_score = np.std(avg_scores)

In [ ]:
fps = stationarity_scores_[stationarity_scores_.stationarity_score > avg_ss_score][stationarity_scores_.stationarity_score < avg_ss_score + std_ss_score].filepath.tolist()

In [ ]:
fps = list(set(fps))

In [ ]:
base_dir = "/om2/data/public/audioset/wavs/eval_segments_downloads/"
save_dir = "/om2/user/bjmedina/data/texture_pairs/spectral_centroid-stationarity-score{}/"

In [ ]:
# index of memory stimulus
k = 0
target_stationarity = -0.6


max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_centroid(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    max_distances.append(max_distance)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]
    
    # # Print results
    # #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    # #print("best segments")

    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)

    # #display(ipd.Audio(norm_s1, rate=samplerate))
    # #display(ipd.Audio(norm_s2, rate=samplerate))

    # if not os.path.exists(save_dir.format(target_stationarity)):
    #     os.makedirs(save_dir.format(target_stationarity))

    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_0.wav", samplerate, norm_s1.astype(np.float32))
    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
x = plt.hist(max_distances, bins=len(max_distances))
plt.axvline(x=np.mean(max_distances), c='r')

In [ ]:
np.sum(np.array(max_distances) > (np.mean(max_distances)) + np.std(max_distances)) / len(max_distances), np.sum(np.array(max_distances) > (np.mean(max_distances)) + np.std(max_distances))

the_bar = np.mean(max_distances) + np.std(max_distances)

In [ ]:
# index of memory stimulus
k = 0

max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0

    # searches for 2s segment that has target stationarity score
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_centroid(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    #max_distances.append(max_distance)

    if max_distance <= the_bar:
        continue
    
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]
    
    # Print results
    #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    #print("best segments")

    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    #display(ipd.Audio(norm_s1, rate=samplerate))
    #display(ipd.Audio(norm_s2, rate=samplerate))

    if not os.path.exists(save_dir.format(target_stationarity)):
        os.makedirs(save_dir.format(target_stationarity))


    ID = fp.split("/")[-1].split(".")[0]

    print(save_dir.format(target_stationarity)+f"{ID}_0.wav")

    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_0.wav", samplerate, norm_s1.astype(np.float32))
    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    print(f"{fp} Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
save_dir = "/om2/user/bjmedina/data/texture_pairs/spectral_contrast-stationarity-score{}/"

In [ ]:
# index of memory stimulus
k = 0
target_stationarity = -0.5


max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    max_distances.append(max_distance)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]
    
    # # Print results
    # #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    # #print("best segments")

    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)

    # #display(ipd.Audio(norm_s1, rate=samplerate))
    # #display(ipd.Audio(norm_s2, rate=samplerate))

    # if not os.path.exists(save_dir.format(target_stationarity)):
    #     os.makedirs(save_dir.format(target_stationarity))

    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_0.wav", samplerate, norm_s1.astype(np.float32))
    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
x = plt.hist(max_distances, bins=len(max_distances))
plt.axvline(x=np.mean(max_distances), c='r')

the_bar = np.mean(max_distances) + np.std(max_distances)

In [ ]:
# index of memory stimulus
k = 0

max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    #max_distances.append(max_distance)

    if max_distance <= the_bar:
        continue
    
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]
    
    # Print results
    #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    #print("best segments")

    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    #display(ipd.Audio(norm_s1, rate=samplerate))
    #display(ipd.Audio(norm_s2, rate=samplerate))

    if not os.path.exists(save_dir.format(target_stationarity)):
        os.makedirs(save_dir.format(target_stationarity))

    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_0.wav", samplerate, norm_s1.astype(np.float32))
    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
# Compute spectral contrast for each segment
spectral_contrast_features = []
for segment in segments:
    S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
    spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time

# Compute Euclidean distances between all pairs
num_segments = len(spectral_contrast_features)
max_distance = -1
best_pair = (0, 1)

for i in range(num_segments):
    for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
        dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
        if dist > max_distance:
            max_distance = dist
            best_pair = (i, j)

# Extract the two most different segments
segment_1 = segments[best_pair[0]]
segment_2 = segments[best_pair[1]]

# Print results
print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")

print("best segments")
display(ipd.Audio(segment_1, rate=samplerate))
display(ipd.Audio(segment_2, rate=samplerate))


In [ ]:
# is the code doing what it is supposed to be doing?
# check other pairs
print(best_pair)
for i in range(num_segments):
    for j in range(i, num_segments):
        if i == j:
            continue
        if i == best_pair[0] and j == best_pair[1]:
            continue
        other_segment_1 = segments[i]
        other_segment_2 = segments[j]
        print(f"segment {i} and {j}")
        display(ipd.Audio(other_segment_1, rate=samplerate))
        display(ipd.Audio(other_segment_2, rate=samplerate))

# what are some selection criteria for screening for discriminable stimuli (so that I don't have to listen to all of them)?

# 1)
# - screen each 10 sec clip for the 2 sec seg that has the stationarity score most similar to -0.6. 
# - get that clip. take the first 0.5s, then take the adjacent 0.5s. (or first 0.5 or some other 0.5)
# - maybe look for the most spectrally different section

# 2) 
# - do same as 1 but chose sound with the lowest stationarity score (or try different scores like -0.5)